In [1]:
# imports
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# LOAD DELIVERABLES (ALREADY LAGGED)
df = pd.read_parquet("data/fx_system_df_lagged.parquet")
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# CONFIG
SHORT_WINDOW = 50
LONG_WINDOW = 200
H = 21
MIN_CROSS_SECTION = 5
ROLL_WINDOW = 252

# FX MAP
FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

# IC SERIES
def compute_ic_series(signal, future_returns):
    ic_vals = []

    for date in signal.index:
        s = signal.loc[date]
        r = future_returns.loc[date]

        valid = s.notna() & r.notna()

        if valid.sum() < MIN_CROSS_SECTION:
            ic_vals.append(np.nan)
            continue

        if s[valid].std() == 0:
            ic_vals.append(np.nan)
            continue

        ic_vals.append(spearmanr(s[valid], r[valid]).statistic)

    return pd.Series(ic_vals, index=signal.index)


# BUILD MEAN REVERSION
def build_mean_reversion_signal(df, fx_map, short, long):
    mr = pd.DataFrame(index=df.index)

    for ccy in fx_map.keys():
        price = df[f"{ccy}_price"]
        log_p = np.log(price)

        ma_short = log_p.rolling(short).mean()
        ma_long = log_p.rolling(long).mean()

        trend = ma_short - ma_long

        # flip → losers = cheap
        mr[ccy] = -trend

    return mr


# FORWARD RETURNS
def compute_forward_returns(returns_df, H):
    return returns_df.rolling(H).sum().shift(-H)


# BUILD SIGNAL
mean_reversion_signal = build_mean_reversion_signal(
    df,
    FX_TICKERS,
    SHORT_WINDOW,
    LONG_WINDOW
)

# CROSS-SECTIONAL NORMALIZATION
rank = mean_reversion_signal.rank(axis=1, pct=True)
mean_reversion_signal = rank.sub(rank.mean(axis=1), axis=0)

# ALIGN + COVERAGE FILTER
common_idx = mean_reversion_signal.index.intersection(returns_df.index)
mean_reversion_signal = mean_reversion_signal.loc[common_idx].copy()
returns_df = returns_df.loc[common_idx].copy()

valid_counts = mean_reversion_signal.notna().sum(axis=1)
mean_reversion_signal = mean_reversion_signal[valid_counts >= MIN_CROSS_SECTION]
returns_df = returns_df.loc[mean_reversion_signal.index]

# DAILY SIGNAL FOR PORTFOLIO CONSTRUCTION
mean_reversion_signal_daily = mean_reversion_signal.copy()

# FORWARD RETURNS
future_returns = compute_forward_returns(returns_df, H)

# NON-OVERLAPPING IC DIAGNOSTIC ONLY
future_returns_ic = future_returns.iloc[::H]

# ALIGN IC INPUTS
common_idx = mean_reversion_signal.index.intersection(future_returns_ic.index)
mean_reversion_signal_ic = mean_reversion_signal.loc[common_idx]
future_returns_ic = future_returns_ic.loc[common_idx]

# IC SERIES
ic_series = compute_ic_series(mean_reversion_signal_ic, future_returns_ic)
ic_clean = ic_series.dropna()

# IC DIAGNOSTICS
expanding_ic_ir = ic_clean.expanding().mean() / ic_clean.expanding().std()
rolling_ic_ir = ic_clean.rolling(ROLL_WINDOW).mean() / ic_clean.rolling(ROLL_WINDOW).std()

# OUTPUT
results = {
    "IC Mean": ic_clean.mean(),
    "IC Median": ic_clean.median(),
    "IC > 0 %": (ic_clean > 0).mean(),
}

results_df = pd.DataFrame([results])

# SAVE DELIVERABLES
mean_reversion_signal_daily.to_parquet("data/mean_reversion_signal.parquet")
ic_series.to_frame("IC").to_parquet("data/mean_reversion_ic_series.parquet")
results_df.to_parquet("data/mean_reversion_summary.parquet")

In [2]:
results_df

,IC Mean,IC Median,IC > 0 %
0,0.023289,0.035714,0.511628
